# HiraBot 3000
### Hiragana-klassificering med PCA & K-Nearest Neighbours

`46 klasser` · `~4 600 bilder` · `32×32 px` · `PCA 95% varians` · `KNN k=1–20`

---

> En svensk turistbyrå, **Visit Lagom**, har en brilliant idé: träna en AI att läsa hiragana perfekt — och sedan smyga in små *"oskyldiga"* meddelanden över hela Japan.
>
> *"Först lär den sig hiragana. Sedan lär den sig slogans. Sedan… bokar hela Japan weekendresor till Dalarna."*

## Pipeline — steg för steg

| Steg | Beskrivning |
|:---:|---|
| **1** | **Ladda bilder** — Gråskala → 32×32 → flatten → normalisera → `x_raw (N, 1024)` |
| **2** | **Split & PCA** — Stratifierad 80/20 split → PCA (95 % varians) → `~170 komponenter` |
| **3** | **Träna KNN** — Testa k = 1–20, välj bästa k, träna slutmodell |
| **4** | **Utvärdera** — Confusion matrix, F1-score, klassificeringsrapport |
| **5** | **Visualisera** — k-kurva, F1 per klass, förväxlingsmatriser, Precision-Recall |

---
## Steg 1 — Ladda datasetet

Alla bilder läses in som gråskala, skalas om till **32 × 32** pixlar, plattas ut till en 1D-vektor med **1 024 features** och normaliseras till intervallet **0–1**.

KNN gör ingen aritmetik på etiketterna, så vanlig heltalskodning via `LabelEncoder` fungerar utan risk för ordningsbias.

In [ ]:
from process_image.image_liquditation import run_dataset_processing
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown

x_raw, y_raw = run_dataset_processing()

display(Markdown(
    f"| | |\n|---|---|\n"
    f"| **Bilder totalt** | {x_raw.shape[0]:,} |\n"
    f"| **Features per bild** | {x_raw.shape[1]:,} |\n"
    f"| **Unika klasser** | {len(set(y_raw))} |\n"
))

In [ ]:
# Visa exempelbilder — en per klass (första 10)
unique_labels = sorted(set(y_raw))
sample_count = min(10, len(unique_labels))

fig, axes = plt.subplots(1, sample_count, figsize=(16, 2.2))

for i, label in enumerate(unique_labels[:sample_count]):
    idx = y_raw.index(label)
    img = x_raw[idx].reshape(32, 32)
    axes[i].imshow(img, cmap='gray_r', interpolation='nearest')
    axes[i].set_title(label, fontsize=11, fontweight='bold', pad=6)
    axes[i].axis('off')

fig.suptitle('Exempelbilder från datasetet', fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

---
## Steg 2 — Train/test-split & PCA

> **Stratifierad split** — 80 % träning / 20 % test. Varje klass proportionellt representerad.

> **PCA — ingen dataläckage** — Anpassas *enbart* på träningsdata, appliceras sedan på båda. Behåller 95 % av variansen (~170 komponenter).

**Varför PCA?** Med 1 024 råa features drabbas KNN av *dimensionsförbannelsen* — avstånd mellan datapunkter blir mindre informativa i högdimensionella rum.

In [ ]:
from data_processing import pca_reduction

x_train_pca, x_test_pca, y_train, y_test, le = pca_reduction(x_raw, y_raw)

reduction = (1 - x_train_pca.shape[1] / x_raw.shape[1]) * 100

display(Markdown(
    f"| | |\n|---|---|\n"
    f"| **Träningsbilder** | {x_train_pca.shape[0]:,} |\n"
    f"| **Testbilder** | {x_test_pca.shape[0]:,} |\n"
    f"| **Features efter PCA** | {x_raw.shape[1]} → {x_train_pca.shape[1]} (−{reduction:.0f} %) |\n"
))

---
## Steg 3 — Hyperparametertuning & KNN-träning

Olika värden på **k** testas (1–20). Det k som ger högst accuracy på testdatan väljs, sedan tränas en slutgiltig KNN-modell med det bästa k-värdet.

In [ ]:
from model_training import train

y_pred, best_k, accuracies = train(x_train_pca, x_test_pca, y_train, y_test)

---
## Steg 4 — Utvärdering

Modellen utvärderas med confusion matrix, F1-score per klass, klassificeringsrapport samt binariserad data för Precision-Recall-kurvor.

In [ ]:
from evaluation import compute_metrics, interpret_results
from settings import K_RANGE

data = {
    "x_test": x_test_pca,
    "y_test": y_test,
    "y_pred": y_pred,
    "le": le,
    "best_k": best_k,
    "accuracies": accuracies,
    "k_range": K_RANGE
}

cm, f1, report, y_test_bin, y_score = compute_metrics(y_test, y_pred, le)
interpret_results(cm, f1, report, data)

# Visa nyckeltal
low_f1 = [c for c, s in zip(le.classes_, f1) if s < 0.8]
display(Markdown(
    f"| Mått | Värde |\n|---|---|\n"
    f"| **Total Accuracy** | {report['accuracy']:.1%} |\n"
    f"| **Macro F1-score** | {report['macro avg']['f1-score']:.3f} |\n"
    f"| **Bästa k** | {best_k} |\n"
    f"| **Klasser med F1 < 0.8** | {len(low_f1)} |\n"
))

---
## Steg 5 — Visualisering

Fyra diagram sammanfattar modellens prestation.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, average_precision_score

COLORS = {
    'primary': '#e94560',
    'dark': '#1a1a2e',
    'accent': '#0f3460',
    'teal': '#00b4d8',
    'grid': '#e0e0e0'
}
plt.rcParams.update({
    'figure.facecolor': '#fafbfe',
    'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#cccccc',
    'axes.labelcolor': COLORS['dark'],
    'xtick.color': '#555555',
    'ytick.color': '#555555',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': COLORS['grid'],
})

### k-Accuracy-kurva

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

k_values = list(data['k_range'])
ax.fill_between(k_values, accuracies, alpha=0.10, color=COLORS['primary'])
ax.plot(k_values, accuracies, marker='o', markersize=7, linewidth=2.5,
        color=COLORS['primary'], markerfacecolor='white', markeredgewidth=2,
        markeredgecolor=COLORS['primary'])

best_idx = k_values.index(best_k)
ax.plot(best_k, accuracies[best_idx], 'o', markersize=14,
        markerfacecolor=COLORS['primary'], markeredgecolor='white',
        markeredgewidth=2.5, zorder=5)
ax.annotate(f'Bästa k = {best_k}\n({accuracies[best_idx]:.3f})',
            xy=(best_k, accuracies[best_idx]),
            xytext=(best_k + 2, accuracies[best_idx] - 0.015),
            fontsize=11, fontweight='bold', color=COLORS['primary'],
            arrowprops=dict(arrowstyle='->', color=COLORS['primary'], lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor=COLORS['primary'], alpha=0.9))

ax.set_xlabel('k (antal grannar)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Accuracy per k-värde', pad=14)
ax.set_xticks(k_values)
plt.tight_layout()
plt.show()

### F1-score per klass

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5.5))

bar_colors = [COLORS['primary'] if s < 0.8 else COLORS['teal'] for s in f1]
ax.bar(le.classes_, f1, color=bar_colors, edgecolor='white', linewidth=0.5, width=0.75)
ax.axhline(y=0.8, color=COLORS['primary'], linestyle='--', linewidth=1.2, alpha=0.6, label='F1 = 0.8')

ax.set_xlabel('Hiraganaklass', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-score', fontsize=12, fontweight='bold')
ax.set_title('F1-score per klass', pad=14)
ax.tick_params(axis='x', rotation=90, labelsize=9)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

### Förväxlingsmatriser

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5),
                                gridspec_kw={'width_ratios': [1.3, 1]})

sns.heatmap(cm, annot=False, cmap='Blues', ax=ax1, cbar_kws={'shrink': 0.8})
ax1.set_title('Förväxlingsmatris (alla 46 klasser)', pad=12)
ax1.set_xlabel('Predikterad klass', fontsize=10)
ax1.set_ylabel('Faktisk klass', fontsize=10)

diag = np.diag(cm)
errors = cm.sum(axis=1) - diag
top_err_idx = np.argsort(errors)[-5:]
cm_focused = cm[np.ix_(top_err_idx, top_err_idx)]
labels_focused = le.classes_[top_err_idx]

sns.heatmap(cm_focused, annot=True, fmt='d', cmap='Reds', ax=ax2,
            xticklabels=labels_focused, yticklabels=labels_focused,
            annot_kws={'fontsize': 12, 'fontweight': 'bold'},
            cbar_kws={'shrink': 0.8}, linewidths=1, linecolor='white')
ax2.set_title('Fokus: topp 5 klasser med flest fel', pad=12)
ax2.set_xlabel('Predikterad klass', fontsize=10)
ax2.set_ylabel('Faktisk klass', fontsize=10)

plt.tight_layout(pad=3)
plt.show()

### Precision-Recall-kurvor

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))

pr_colors = ['#e94560', '#0f3460', '#00b4d8', '#d45e00', '#2e7d32']
n_classes = y_test_bin.shape[1]

for i in range(min(5, n_classes)):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_score[:, i])
    ax.plot(recall, precision, linewidth=2.5, color=pr_colors[i % len(pr_colors)],
            label=f'{le.classes_[i]}  (AP = {ap:.2f})')

ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('Precision-Recall-kurvor (topp 5 klasser)', pad=14)
ax.legend(fontsize=11, loc='lower left', framealpha=0.95,
          edgecolor='#cccccc', fancybox=True)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
plt.tight_layout()
plt.show()

---
## Tolkning av resultat

### Förväxlingsmatris

- **Diagonalen** — korrekta förutsägelser; starkare färg = bättre.
- **Utanför diagonalen** — förväxlingar; rad = faktisk, kolumn = predikterad.
- **Fokusmatris** — isolerar de 5 svåraste klasserna.

### Utvärderingsmått

| Mått | Beskrivning |
|---|---|
| **Accuracy** | Andel korrekta av totala |
| **Precision** | Hur pålitlig modellen är per klass |
| **Recall** | Hur mycket av klassen hittas |
| **F1** | Harmoniskt medelvärde av precision & recall |
| **AP** | Sammanfattar PR-kurvan (närmare 1 = bättre) |

### Analys i KNN-kontext

> **Grafisk likhet** — tecken kan vara för lika vid 32×32.
>
> **PCA** — detaljer kan gå förlorade under dimensionsreducering.
>
> **k-värdet** — för litet k → brusigt; för stort k → suddiga beslutsgränser.

### Rekommenderade åtgärder vid systematiska förväxlingar

1. Undersök bildkvaliteten för de specifika tecknen.
2. Justera PCA-komponenter för att behålla mer varians.
3. Testa andra k-värden för just de problematiska klasserna.

---
## Teknikstack & designbeslut

| Bibliotek | Användning |
|---|---|
| **scikit-learn** | KNN, PCA, LabelEncoder, metrics |
| **NumPy** | Arrayhantering |
| **Pillow** | Bildladdning & förbehandling |
| **Matplotlib / Seaborn** | Visualisering |

**Designbeslut:**

- **KNN** — enkelt, snabbt, ingen traditionell träningsfas
- **PCA före KNN** — motverkar dimensionsförbannelsen
- **PCA fittas på train** — förhindrar dataläckage
- **Stratifierad split** — rättvis klassrepresentation
- **Settings-modul** — centraliserade parametrar